In [1]:
import numpy as np
import pandas as pd
import rioxarray
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from affine import Affine
from libpysal.weights import DistanceBand
import esda
from pathlib import Path
import os
root = Path.cwd()

c:\Users\roseh\miniconda3\envs\urban-tree-canopy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
print({k: v for k, v in os.environ.items()
       if any(s in k.upper() for s in ('PROXY','SSL_CERT','CA_BUNDLE','CURL'))})

{'CURL_CA_BUNDLE': 'C:\\Program Files\\PostgreSQL\\17\\ssl\\certs\\ca-bundle.crt', 'SSL_CERT_DIR': 'C:\\Users\\roseh\\miniconda3\\envs\\urban-tree-canopy\\Library\\ssl\\certs', 'SSL_CERT_FILE': 'C:\\Users\\roseh\\miniconda3\\envs\\urban-tree-canopy\\Library\\ssl\\cacert.pem', '__CONDA_OPENSSL_CERT_DIR_SET': '1', '__CONDA_OPENSSL_CERT_FILE_SET': '1'}


In [2]:
import os
print({k: v for k, v in os.environ.items()
       if any(s in k.upper() for s in ('PROXY','SSL_CERT','CA_BUNDLE','CURL'))})

{'CURL_CA_BUNDLE': 'C:\\Program Files\\PostgreSQL\\17\\ssl\\certs\\ca-bundle.crt', 'SSL_CERT_DIR': 'C:\\Users\\roseh\\miniconda3\\envs\\urban-tree-canopy\\Library\\ssl\\certs', 'SSL_CERT_FILE': 'C:\\Users\\roseh\\miniconda3\\envs\\urban-tree-canopy\\Library\\ssl\\cacert.pem', '__CONDA_OPENSSL_CERT_DIR_SET': '1', '__CONDA_OPENSSL_CERT_FILE_SET': '1', 'GDAL_CURL_CA_BUNDLE': 'c:\\Users\\roseh\\miniconda3\\envs\\urban-tree-canopy\\Lib\\site-packages\\certifi\\cacert.pem', 'PROJ_CURL_CA_BUNDLE': 'c:\\Users\\roseh\\miniconda3\\envs\\urban-tree-canopy\\Lib\\site-packages\\certifi\\cacert.pem'}


In [1]:
import pyproj
pyproj.__file__

'C:\\Users\\roseh\\AppData\\Roaming\\Python\\Python312\\site-packages\\pyproj\\__init__.py'

In [2]:
import pyproj
pyproj.__file__

'c:\\Users\\roseh\\miniconda3\\envs\\urban-tree-canopy\\Lib\\site-packages\\pyproj\\__init__.py'

In [1]:
import zarr
zarr.__version__

'2.18.7'

In [2]:
def load_residuals(pred_path, lidar_path):
    """10 m residual (pred - lidar) and lidar canopy, both NaN wherever either raster is invalid."""
    pred = rioxarray.open_rasterio(pred_path, masked=True).squeeze('band', drop=True)
    lidar = rioxarray.open_rasterio(lidar_path, masked=True).squeeze('band', drop=True)
    assert pred.shape == lidar.shape and pred.rio.transform().almost_equals(lidar.rio.transform()), \
        'prediction and lidar rasters are not on the same grid'
    valid = pred.notnull() & lidar.notnull()
    resid = (pred - lidar).where(valid)
    return resid, lidar.where(valid)


def canopy_bins(canopy, width=5):
    """Bin index per pixel: 0 = exactly 0%, last = exactly 100%, otherwise `width`-point bins."""
    n_mid = int(np.ceil(100 / width)) ## there are 20 5-point bins between 0 and 100
    b = 1 + np.clip(np.floor(canopy / width), 0, n_mid - 1) # makes bins 1 - 20
    b = np.where(canopy <= 0, 0, b)  # 0% canopy gets bin 0
    b = np.where(canopy >= 100, n_mid + 1, b)   # 100% canopy gets bin 21
    return np.where(np.isfinite(canopy), b, -1).astype(int)  # 2D array where each pixel is the bin number

## must run after aggregation. running before adds artifical spatial structure not present in residuals.
def canopy_adjust(resid, lidar, width=5):
    """Subtract the citywide mean residual of each canopy bin. Returns (adjusted DataArray, bin df)."""
    r, b = resid.values, canopy_bins(lidar.values, width)
    ok = b >= 0  # exclude bin -1 that contains invalid pixels
    ## get mean residual, min canopy, max canopy and n pixels per  bin 
    table = (pd.DataFrame({'bin': b[ok], 'resid': r[ok], 'canopy': lidar.values[ok]})
               .groupby('bin').agg(canopy_lo=('canopy', 'min'), canopy_hi=('canopy', 'max'),
                                   n=('resid', 'size'), mean_resid=('resid', 'mean'))
               .reset_index())
    lookup = np.full(b.max() + 2, np.nan)
    lookup[table['bin'].to_numpy()] = table['mean_resid'].to_numpy() # 1D array mapping bin numbers to mean residuals
    adj = np.where(ok, r - lookup[np.where(ok, b, 0)], np.nan) # substract mean residual according to bin number
    return resid.copy(data=adj.astype('float32')), table ## wrap output in same data array as input residuals


def correlogram(arr, distances, res=10.0, n_pairs=300_000, seed=42):
    """Autocorrelation of a 2-D array at each distance, from random pixel pairs.

    For each distance d: draw n_pairs valid pixels, step d metres in a random direction, keep pairs where
    both pixels are valid, and compute mean((a - mu)(b - mu)) / var. This is the Moran's I-type
    autocorrelation for that distance class (1 = identical, 0 = unrelated).
    """
    rng = np.random.default_rng(seed)
    ok = np.isfinite(arr)
    rows, cols = np.nonzero(ok)
    vals = arr[ok].astype('float64')
    mu, var = vals.mean(), vals.var()
    H, W = arr.shape
    out = []
    for d in distances:
        i = rng.integers(len(vals), size=n_pairs)
        th = rng.uniform(0, 2 * np.pi, n_pairs)
        dr = np.rint(d / res * np.sin(th)).astype(int)
        dc = np.rint(d / res * np.cos(th)).astype(int)
        keep = (dr != 0) | (dc != 0)
        r2, c2 = rows[i] + dr, cols[i] + dc
        keep &= (r2 >= 0) & (r2 < H) & (c2 >= 0) & (c2 < W)
        a = vals[i[keep]]
        b = arr[r2[keep], c2[keep]].astype('float64')
        good = np.isfinite(b)
        a, b = a[good], b[good]
        dist = np.hypot(dr[keep][good], dc[keep][good]) * res
        out.append({'distance_m': d, 'mean_pair_distance_m': dist.mean(), 'n_pairs': len(a),
                    'autocorrelation': np.mean((a - mu) * (b - mu)) / var,
                    'covariance': np.mean((a - mu) * (b - mu))}) 
    return pd.DataFrame(out)

def plot_correlogram(corrs):
    """corrs: {label: correlogram DataFrame}."""
    fig, ax = plt.subplots(figsize=(6, 3.8))
    styles = [dict(color='black', ls='-'), dict(color='gray', ls='--')]
    
    ax.plot(corrs['mean_pair_distance_m'], corrs['autocorrelation'], marker='o', ms=4, lw=1.5, color='black',ls='-')
    ax.axhline(0, color='0.5', lw=0.8)
    ax.set_xscale('log')
    ticks = [10, 30, 100, 300, 1000, 3000]
    ax.set_xticks(ticks); ax.set_xticklabels([f'{t:,}' for t in ticks]); ax.minorticks_off()
    ax.set_xlabel('Distance between pixels (m)')
    ax.set_ylabel('Autocorrelation of 10 m residuals')
    ax.grid(axis='y', alpha=0.3)
    # for s in ('top', 'right'):
    #     ax.spines[s].set_visible(False)
    plt.tight_layout()
    #plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.show()

def _coarse_coords(src, factor, nx, ny):
    tf = src.rio.transform()
    new_tf = tf * Affine.scale(factor)
    x = new_tf.c + (np.arange(nx) + 0.5) * new_tf.a
    y = new_tf.f + (np.arange(ny) + 0.5) * new_tf.e
    return x, y, new_tf


def aggregate(da, cell_size, min_coverage=0.75, res=10):
    """Mean of valid 10 m pixels per coarse cell; cells below min_coverage set to NaN."""
    
    f = int(cell_size // res)
    kw = dict(x=f, y=f, boundary='pad')
    s = da.fillna(0).coarsen(**kw).sum()
    n = da.notnull().astype('float32').coarsen(**kw).sum()
    out = (s / n).where(n >= min_coverage * f * f).astype('float32')
    x, y, tf = _coarse_coords(da, f, out.sizes['x'], out.sizes['y'])
    return (out.assign_coords(x=x, y=y).rio.write_crs(da.rio.crs)
               .rio.write_transform(tf).rio.write_nodata(np.nan, encoded=False))

In [7]:
import os


features = "three_seasons"
out_path = root / 'model_outputs' / 'residuals' / features
cell_size = 100

os.makedirs(out_path,exist_ok=True)

pred_path = root / 'model_outputs' / features / f'nyc21_canopy_from_nyc21_{features}_OOF_model.tif'
lidar_path = root / 'data' / 'nyc21' / f'nyc21_tree_canopy_lidar.tif'

resid, lidar = load_residuals(pred_path, lidar_path)

## adjusting first over-adjusts residuals and changes spatial structure of residuals
#adj, bins = canopy_adjust(resid, lidar)
#bins.to_csv(out_path / f'{output_label}_canopy_bias_by_bin.csv', index=False)
# corr_distances = [10, 20, 30, 50, 75, 100, 150, 200, 300, 500, 750, 1000, 1500, 2000, 3000]
# corr_df = correlogram(arr=resid.values,distances=corr_distances)  ## Moran's I: how much autocorrelation present in data
# corr_df.to_csv(out_path / f'nyc21_{features}_correlogram.csv')
# plot_correlogram(corr_df)

In [8]:
#cell_size = 100    # choose scale the removes pixel-level noise but preserves correlation structure

cell_resid = aggregate(resid,cell_size=cell_size)
cell_lidar = aggregate(lidar,cell_size=cell_size).where(cell_resid.notnull())
cell_adjust, bins = canopy_adjust(cell_resid, cell_lidar) ## adjust residuals after aggregating
bins.to_csv(out_path / f'{features}_canopy_bias_by_bin_{cell_size}m.csv', index=False)
cell_adjust.rio.to_raster(out_path / f'{features}_resid_adj_{cell_size}m.tif', compress='lzw')

In [9]:
distance_band = 1.5 * cell_size
permutations = 9999

df, glob = local_statistics(cell_adjust,neighbour_m=distance_band,permutations=permutations,alpha=0.05)
print(f"Global Moran's I at {cell_size}: {glob.I:.3f} (pseudo p = {glob.p_sim:.4f})")
df.to_csv(out_path / f'{features}_local_stats_{cell_size}m.csv', index=False)
to_raster(df, cell_adjust, 'lisa_cluster', f'{features}_lisa_clusters_{cell_size}m.tif', dtype='uint8', nodata=255)
#to_raster(df, cells, 'gi_z', f'{output_label}_gi_z_{cell_size}m.tif')
summ = cluster_summary(df)
summ.to_csv(out_path / f'{features}_lisa_summary_{cell_size}m.csv')


Global Moran's I at 100: 0.432 (pseudo p = 0.0001)


In [6]:

## control family-wise error rate
def _bh_significant(p, alpha=0.05):
    """True where p is significant at false-discovery rate alpha."""
    p = np.asarray(p)
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    passed = np.nonzero(ranked <= alpha)[0]
    sig = np.zeros(len(p), bool)
    if passed.size:
        sig[order[:passed.max() + 1]] = True
    return sig

CLUSTER_NAMES = {0: 'Not significant', 1: 'High-High', 2: 'Low-High', 3: 'Low-Low', 4: 'High-Low'}
def local_statistics(cells, neighbour_m, permutations, alpha, seed=42):
    """LISA for every valid coarse cell. Returns (per-cell DataFrame, global Moran's I object)."""
    
    df = cells.to_series().dropna().rename('resid_adj').reset_index()

    def weights():
        return DistanceBand(df[['x', 'y']].to_numpy(), threshold=neighbour_m, binary=True, silence_warnings=True)
    w = weights()
    if w.islands:                                   # cells with no valid neighbours cannot be tested
        df = df.drop(index=w.islands).reset_index(drop=True)
        w = weights()
    w_bin = weights()                               # binary copy for Gi* (which adds the cell itself)
    w.transform = 'r'                               # row-standardised for Moran's I
    y = df['resid_adj'].to_numpy('float64')

    glob = esda.Moran(y, w, permutations=permutations)
    lisa = esda.Moran_Local(y, w, permutations=permutations, seed=seed, n_jobs=-1,keep_simulations=False)
    #gi = esda.G_Local(y, w_bin, transform='B', permutations=permutations, star=True, seed=seed,keep_simulations=False,n_jobs=1)

    df['n_neighbours'] = [w.cardinalities[i] for i in range(len(df))]
    df['lisa_I'] = lisa.Is
    df['lisa_quadrant'] = lisa.q                      # 1 HH, 2 LH, 3 LL, 4 HL (relative to the citywide mean)
    df['lisa_p'] = lisa.p_sim
    df['lisa_sig_fdr'] = _bh_significant(lisa.p_sim, alpha)
    df['lisa_cluster'] = np.where(df['lisa_sig_fdr'], df['lisa_quadrant'], 0)
    df['lisa_label'] = df['lisa_cluster'].map(CLUSTER_NAMES)

    # df['gi_z'] = gi.Zs                                # > 0: neighbourhood of over-prediction; < 0: under-prediction
    # df['gi_p'] = gi.p_sim
    # df['gi_sig_fdr'] = _bh_significant(gi.p_sim, alpha)
    # df['gi_spot'] = np.select([df['gi_sig_fdr'] & (df['gi_z'] > 0), df['gi_sig_fdr'] & (df['gi_z'] < 0)],
    #                           ['Hot (over-prediction)', 'Cold (under-prediction)'], 'Not significant')
    return df, glob


def to_raster(df, template, column, path, dtype='float32', nodata=np.nan):
    """Write one per-cell column back onto the coarse grid."""
    arr = np.full(template.shape, nodata, dtype=dtype)
    r = np.searchsorted(-template['y'].values, -df['y'].values)     # y decreases down the rows
    c = np.searchsorted(template['x'].values, df['x'].values)
    arr[r, c] = df[column].to_numpy(dtype)
    template.copy(data=arr).rio.write_nodata(nodata, encoded=False).rio.to_raster(out_path / path, compress='lzw')


def cluster_summary(df):
    """Cells per LISA cluster type, with their mean adjusted residual (the effect size)."""
    return (df.groupby('lisa_label')
              .agg(n_cells=('resid_adj', 'size'), mean_resid_adj=('resid_adj', 'mean'))
              .assign(share_of_cells=lambda t: t['n_cells'] / t['n_cells'].sum())
              .reindex([v for v in CLUSTER_NAMES.values()]).dropna(how='all'))

Global Moran's I at 100: 0.427 (pseudo p = 0.0001)
                 n_cells  mean_resid_adj  share_of_cells
lisa_label                                              
Not significant    68896          -1.214           0.901
High-High           3295           6.879           0.043
Low-High             244          -3.244           0.003
Low-Low             3764          -8.539           0.049
High-Low             277           0.935           0.004


In [11]:
(df['lisa_p'] < 0.05).mean().round(3)

np.float64(0.29)

In [ ]:
import xarray as xr
import rioxarray

def load(path):
    return rioxarray.open_rasterio(path, masked=True).squeeze('band', drop=True)

prefix = {'2018':'nyc18', '2021':'nyc21'}
in_path = root / 'model_outputs' / 'residuals'
cl = {y: load(in_path / f'{p}_lisa_clusters_100m.tif') for y, p in prefix.items()}     # 0 ns, 1 HH, 2 LH, 3 LL, 4 HL
ra = {y: load(in_path / f'{p}_resid_adj_100m.tif') for y, p in prefix.items()}
cl['2018'], cl['2021'], ra['2018'], ra['2021'] = xr.align(cl['2018'], cl['2021'], ra['2018'], ra['2021'], join='exact')

both = cl['2018'].notnull() & cl['2021'].notnull()                       # cells tested in both years
a, b = cl['2018'].values[both.values], cl['2021'].values[both.values]
names = {0: 'Not significant', 1: 'High-High', 2: 'Low-High', 3: 'Low-Low', 4: 'High-Low'}

# 1. agreement
r = np.corrcoef(ra['2018'].values[both.values], ra['2021'].values[both.values])[0, 1]
print(f'correlation of adjusted cell residuals, 2018 vs 2021: r = {r:.2f}')
print(pd.crosstab(pd.Series(a).map(names).rename('2018'), pd.Series(b).map(names).rename('2021')))
for code in (1, 3):
    overlap = (a[b == code] == code).mean()       # share of 2021 cluster cells with the same type in 2018
    chance = (a == code).mean()                   # share expected if the two years were unrelated
    print(f'{names[code]}: {overlap:.0%} also in 2018 (chance: {chance:.0%})')

# 2. persistence categories (written as a raster for mapping)
cat = np.select([(cl['2018'] == 1) & (cl['2021'] == 1),
                 (cl['2018'] == 3) & (cl['2021'] == 3),
                 (cl['2018'] > 0) | (cl['2021'] > 0)],
                [1, 2, 3], 0)                    # 1 persistent over, 2 persistent under, 3 one year only, 0 ns
persist = cl['2021'].copy(data=np.where(both, cat, 255).astype('uint8'))


In [8]:
persist.rio.to_raster(in_path / 'persistent_clusters_100m.tif', compress='lzw')